## Issue 117 (Purav): PETai split repair, fragment remap and leakage-free re-split

Lead     : `Purav Gupta`

Issue    : [Github Issue #117](https://github.com/petadex/igem-toronto/issues/117) _fix the PETai split so the model can be evaluated_

Start    : `2026-10-15`

Complete : `2026-10-15`

Files    : `resources/261015_issue117_petai_split_repair/` scripts and audit (committed); releases under `/project/6122149/purav/petai-labels/` (`combined` is v1, `remapped` is step 1, `v2` is the result)

## Introduction

Issue 116 published the PETai activity label set and measured two defects that make it unusable for evaluation. The test fold held 7 negative labels on 5 of 170 sequences, because the halo screen that supplies 168 of the 200 negatives went 178 train, 12 val, 0 test. Label provenance was inverted, with train at 19.6% LLM against test at 62.9%, and the 4,898 LLM labels contain no negatives at all because papers report successes.

This issue repairs both. The plan was a fragment remap followed by a stratified re-split on 30% clusters, and the remap went exactly as predicted. The re-split did not, and the reason is worth stating up front because it changes the modelling target.

The release ships three clusterings that are not nested. Splitting on `cluster_30` alone leaves sequences that share a 50% identity cluster sitting in different folds. Once the split unit is corrected to something that is clean at every threshold, 48% of the sequences collapse into a single homology component that carries 181 of the 200 negatives and 187 of the 197 halo labels. Binary halo activity, which the handoff brief named as the clean choice for classification, cannot be evaluated without homology leakage. There are not enough homology-independent inactive sequences in the data, because the negatives and the actives are all one tight family of PET hydrolases.

What does survive is regression. PET monomer release in concentration units gives 1,967 labels, and 256 of them sit outside that component on 61 sequences across 26 components, 203 of those human-curated. So the evaluated result is regression, halo activity becomes train-only, and the split is clean at 30%, 50% and 90% identity simultaneously.

## Objectives

1. Remap the 73 contained fragments onto their parents, writing a new release rather than editing v1 in place.
2. Establish whether the shipped clusterings are nested, since the whole homology guarantee rests on it.
3. Re-split with stratification, and measure honestly what each candidate costs in leakage.
4. Determine which target can be evaluated under strict homology control, and split on that.
5. Audit the result from scratch, re-deriving every claim rather than trusting the writer.

### Step 0: setup

Everything here is stdlib only. The release is six CSV and FASTA files and none of the work needs pandas, which matters because the system python has neither pandas nor `datasets` without loading modules first.

In [ ]:
import collections, csv, json, subprocess
from pathlib import Path

LABELS   = Path("/project/6122149/purav/petai-labels")
V1       = LABELS / "combined"
REMAPPED = LABELS / "remapped"
V2       = LABELS / "v2"
RES      = Path("resources/261015_issue117_petai_split_repair")

def read_csv(path, name):
    with (path / name).open(newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

v1_labels    = read_csv(V1, "labels.csv")
v1_sequences = read_csv(V1, "sequences.csv")
v1_split     = {r["seq_md5"]: r["split"] for r in read_csv(V1, "split.csv")}

print(f"v1: {len(v1_labels):,} labels on {len(v1_sequences):,} sequences, "
      f"{len(v1_split):,} of them assigned to a fold")

### Step 1: remap the contained fragments

82 sequences carry labels but no fold assignment, stranding 3,188 labels, which is 24.4% of the release. 73 of them have a `contained_in` parent that already sits in a fold, so their labels can be reattached to the parent instead of discarded. The remaining 9 have no parent and are lost with their 38 labels.

Confirm the arithmetic before moving anything.

In [ ]:
labels_per_seq = collections.Counter(row["seq_md5"] for row in v1_labels)
unsplit = [s for s in v1_sequences if not v1_split.get(s["seq_md5"])]

recoverable = [s for s in unsplit if s["contained_in"] and v1_split.get(s["contained_in"])]
lost        = [s for s in unsplit if s not in recoverable]

def total(seqs):
    return sum(labels_per_seq[s["seq_md5"]] for s in seqs)

print(f"unsplit      {len(unsplit):>4} sequences  {total(unsplit):>5,} labels")
print(f"recoverable  {len(recoverable):>4} sequences  {total(recoverable):>5,} labels "
      f"onto {len({s['contained_in'] for s in recoverable})} parents")
print(f"lost         {len(lost):>4} sequences  {total(lost):>5,} labels")

73 fragments, 3,150 labels, 64 parents, and 9 sequences with 38 labels unrecoverable. Every figure matches the handoff brief.

`remap_fragments.py` performs the move. It writes a new release and never touches v1. Lineage is preserved rather than discarded: each remapped label keeps its original md5 in `orig_seq_md5` and carries a `remapped` flag, so a fragment-level analysis is still possible later, which step 7 of the plan will need for the truncation ablation.

In [ ]:
print(subprocess.run(
    ["python3", "-I", str(RES / "remap_fragments.py"), str(V1), str(REMAPPED)],
    capture_output=True, text=True, check=True).stdout)

1,137 sequences and 13,054 labels, with all 200 negatives intact, since none of the 9 dropped sequences carried one. 1,314 PET labels and 773 melting labels come back.

One correction to the brief. It reports the recovered PET measurements as 1,301 and the full release as 4,492 PET labels on 449 sequences. Neither figure reproduces under any definition tried here: exact `substrate == "PET"` gives 4,287 on 415 sequences and 1,281 recovered, and treating `substrate` as a semicolon list and matching the `PET` token gives 4,595 on 426 and 1,314 recovered. Neither of the two profiling scripts counts PET at all, so the figure's provenance is unclear. The token definition is used from here on and stated wherever it appears.

The split is deliberately untouched at this stage, so test still holds 7 negatives. That is step 2.

### Step 2a: are the clusterings nested?

The plan was to re-split on 30% clusters. That guarantees a homology-clean split only if a 30% cluster contains the 50% and 90% clusters of its members. The release ships all three, so this is checkable rather than assumable, and it had not been checked.

In [ ]:
remapped_seqs = read_csv(REMAPPED, "sequences.csv")
LEVELS = ("cluster_90", "cluster_50", "cluster_30")

for level in LEVELS:
    print(f"{level}: {len({s[level] for s in remapped_seqs}):,} groups")

for fine, coarse in (("cluster_90", "cluster_50"), ("cluster_50", "cluster_30")):
    straddle = collections.defaultdict(set)
    for s in remapped_seqs:
        straddle[s[fine]].add(s[coarse])
    bad = [k for k, v in straddle.items() if len(v) > 1]
    print(f"{fine} groups spanning more than one {coarse}: {len(bad)} of {len(straddle)}")

The group counts are in the right order, 613 at 90% identity down to 196 at 30%, so `cluster_30` is the coarsest as expected. But 14 of the 318 50% clusters straddle two different 30% clusters. The three clusterings were produced by independent runs rather than a cascade.

The consequence is that splitting on `cluster_30` leaves pairs of sequences that share a 50% identity cluster, and are therefore more similar than the 30% criterion implies, in different folds. The v1 release has four such groups. Its stated guarantee, homology-clean at 30% identity with the leakage check passing, is true for `cluster_30` and incomplete.

In [ ]:
def leakage(seqs, split_of):
    import itertools
    out = {}
    for level in LEVELS:
        groups = collections.defaultdict(list)
        for s in seqs:
            if split_of.get(s["seq_md5"]):
                groups[s[level]].append(s["seq_md5"])
        spanning, pairs, involved = 0, 0, set()
        for members in groups.values():
            folds = {split_of[m] for m in members}
            if len(folds) > 1:
                spanning += 1
                for a, b in itertools.combinations(members, 2):
                    if split_of[a] != split_of[b]:
                        pairs += 1
                        involved.update((a, b))
        out[level] = (spanning, len(groups), pairs, len(involved))
    return out

print("v1 release, cross-fold leakage by threshold:")
for level, (span, n, pairs, inv) in leakage(v1_sequences, v1_split).items():
    print(f"  {level}: {span} of {n} groups span folds, "
          f"{pairs:,} cross-fold pairs over {inv} sequences")

### Step 2b: the stratified 30% split, and why it was rejected

`resplit_frontier.py` searches for the best stratified split using `cluster_30` as the unit, as the plan specified. All 200 negatives sit in 13 of the 196 clusters, and 160 of them in the four largest clusters in the set, so inactive count in test trades directly against fold size. The 13 negative-bearing clusters are enumerated exhaustively over the three folds and the remaining 183 filled greedily, with the frontier reported at a range of test inactive floors so the choice is explicit.

The best point looks like a complete fix. Test reaches 170 sequences at 15.0%, with 54 negative labels on 53 sequences, and the LLM share of test falls from 62.9% to 16.3% while train rises to 43.2%. Both blockers resolved.

Then measure what it costs.

In [ ]:
frontier = json.loads((RES / "frontier_cluster30.json").read_text())
chosen   = next(f for f in frontier if f["floor_test"] == 50)

remapped_split = {s["seq_md5"]: chosen["assignment"][s["cluster_30"]]
                  for s in remapped_seqs}

print("best stratified cluster_30 split, cross-fold leakage by threshold:")
for level, (span, n, pairs, inv) in leakage(remapped_seqs, remapped_split).items():
    print(f"  {level}: {span} of {n} groups span folds, "
          f"{pairs:,} cross-fold pairs over {inv} sequences")

13,494 cross-fold pairs at 50% identity or better across 430 sequences, and 108 pairs at 90% or better across 37. Against v1's 42 pairs over 21 sequences and none at 90%, optimising for test inactives made leakage an order of magnitude worse. 37 sequences in cross-fold near-duplicate pairs would inflate any metric computed on this split, so it is rejected. The frontier is kept in `frontier_cluster30.json` as the record of what was tried.

### Step 2c: the leakage-free unit, and the component that eats the dataset

The unit that is clean at every threshold is a connected component of the graph linking any two sequences that share a cluster at any level. `components.py` builds it.

In [ ]:
print(subprocess.run(
    ["python3", "-I", str(RES / "components.py"), str(REMAPPED / "sequences.csv"),
     str(RES / "components.csv")],
    capture_output=True, text=True, check=True).stdout)

196 clusters become 184 components, and one of them holds 546 sequences, 48% of the release. The 12 merges chained the large clusters together.

In [ ]:
component = {r["seq_md5"]: r["component"]
             for r in read_csv(RES, "components.csv")}
remapped_labels = read_csv(REMAPPED, "labels.csv")

CONC = {"µM", "mM", "nM", "M", "µmol/L", "mM/L"}
tokens    = lambda row: [x.strip() for x in row["substrate"].split(";")]
is_target = lambda row: ("PET" in tokens(row) and row["task"] == "quantitative"
                         and row["unit"] in CONC)

by_component = collections.defaultdict(collections.Counter)
for row in remapped_labels:
    stat = by_component[component[row["seq_md5"]]]
    stat["labels"] += 1
    stat["negative"] += row["is_negative"] == "True"
    stat["halo"]     += row["source"] == "halo"
    stat["target"]   += is_target(row)
for s in remapped_seqs:
    by_component[component[s["seq_md5"]]]["sequences"] += 1

blob = max(by_component, key=lambda c: by_component[c]["sequences"])
tot  = collections.Counter()
for stat in by_component.values():
    tot.update(stat)

b = by_component[blob]
print(f"largest component: {b['sequences']} sequences "
      f"({100 * b['sequences'] / tot['sequences']:.1f}%), "
      f"{b['labels']:,} labels ({100 * b['labels'] / tot['labels']:.1f}%)")
for key in ("negative", "halo", "target"):
    print(f"  {key:>8}: {b[key]:>4} of {tot[key]:>4} inside it, "
          f"{tot[key] - b[key]:>3} outside")

This is the result that reframes the issue. The component carries 181 of the 200 negatives and 187 of the 197 halo binary labels. Only 19 negatives and 10 halo labels sit anywhere else. At 48% of sequences the component cannot be an evaluation fold, so it has to be train, and binary halo activity therefore has no evaluable test set at all under strict homology control.

Before accepting that, check whether the component is held together by a few breakable links. `bridge_analysis.py` enforces nesting by dropping the minority members of every straddling group and recomputes.

It is not breakable cheaply. One 50% cluster alone spans five of the large 30% clusters and contributes 196 sequences, and the six largest clusters are linked by a dense mesh of independent groups rather than a chain with cut points. Forcing nesting costs 211 sequences and 32.6% of the labels. Biologically this is unsurprising, since these are all PET hydrolases and cutinases and the halo screen sampled exactly that family.

So the classification target is not rescuable. The regression target is.

In [ ]:
target = [row for row in remapped_labels if is_target(row)]
outside = [row for row in target if component[row["seq_md5"]] != blob]
human  = [row for row in outside if row["source"] != "llm"]

print(f"PET concentration labels: {len(target):,} total, "
      f"{len(outside)} outside the train component "
      f"on {len({r['seq_md5'] for r in outside})} sequences "
      f"across {len({component[r['seq_md5']] for r in outside})} components")
print(f"  of those, {len(human)} are human-curated on "
      f"{len({r['seq_md5'] for r in human})} sequences "
      f"across {len({component[r['seq_md5']] for r in human})} components")
print("  sources:", collections.Counter(r["source"] for r in outside).most_common())

256 evaluable labels outside the component, 203 of them human-curated across 18 components. Enough for a real held-out fold, and the provenance problem solves itself here: the regression target is 1,743 `activity` labels against 224 LLM, so restricting evaluation to human-curated labels costs little and removes the unvalidated extractions from every metric.

Units are the one caveat. `mmol` and `µmol` are amounts rather than concentrations and are not interconvertible without an assay volume, so they are excluded from the target and left for step 3 of the plan. The target is restricted to µM, mM, nM and µmol/L.

### Step 2d: write the split

`split_components.py` assigns the folds. The train component is forced to train, every component holding a halo label is forced to train so halo is train-only by construction, and the 18 components carrying evaluable labels are dealt into val and test by exhaustive enumeration, balancing evaluated labels and evaluated sequences at once. Everything else goes to train.

In [ ]:
print(subprocess.run(
    ["python3", "-I", str(RES / "split_components.py"), str(REMAPPED),
     str(RES / "components.csv"), str(V2)],
    capture_output=True, text=True, check=True).stdout)

### Step 3: audit

`audit_v2.py` re-derives every claim from the two releases independently of the scripts that wrote them. It checks that v1 is byte-identical to what issue 116 published, that label counts conserve, that no fragment survived the remap, that all 200 negatives are present, that no group at any of the three thresholds spans folds, that `split.csv` agrees with `sequences.csv`, that the evaluated set contains no LLM labels and val and test contain no halo labels, and that the FASTA matches the sequence table exactly.

In [ ]:
print(subprocess.run(
    ["python3", "-I", str(RES / "audit_v2.py"), str(V1), str(V2), str(RES / "audit.json")],
    capture_output=True, text=True, check=True).stdout)

### Step 3: normalise units and name the targets

The release spells units 621 different ways across 13,054 labels. Some of that is encoding rather than substance, so measure how much before writing any mapping.

In [ ]:
import re, unicodedata

DASHES = dict.fromkeys(map(ord, "\u2212\u2013\u2014\u2010\u2011"), "-")

def unit_key(raw):
    u = unicodedata.normalize("NFKC", raw).translate(DASHES)
    return re.sub(r"\s+", " ", u).strip()

v2_labels = read_csv(V2, "labels.csv")
raw   = {row["unit"] for row in v2_labels}
folded = {unit_key(u) for u in raw}
lowered = {unit_key(u).lower() for u in raw}
print(f"raw spellings {len(raw)}, after NFKC and dash folding {len(folded)}, "
      f"if lowercased {len(lowered)}")

# Lowercasing looks tempting and is wrong.
collisions = collections.defaultdict(set)
for u in raw:
    collisions[unit_key(u).lower()].add(unit_key(u))
unsafe = {k: v for k, v in collisions.items() if len(v) > 1}
print("case-only pairs that do not mean the same thing:")
for k, v in sorted(unsafe.items()):
    if k in {"nm", "mm", "μm"}:
        counts = {s: sum(1 for row in v2_labels if unit_key(row["unit"]) == s) for s in v}
        print(f"  {k!r}: {counts}")

NFKC folding plus dash unification takes 621 spellings to 590, merging the micro sign U+00B5 with Greek mu and the superscript minus with a hyphen. Lowercasing would take it to 573 and must not be applied: `nM` and `nm` are nanomolar and nanometre, `mM` and `mm` are millimolar and millimetre, and `µM` and `µm` are micromolar and micrometre. All six occur, because the release records both concentrations and absorbance wavelengths.

The remaining 590 are not a normalisation problem but a target-definition problem. `normalise_units.py` names eight targets, each with one canonical unit and an explicit map of the spellings that convert into it. Anything that does not map is left unassigned rather than coerced, and anything suspicious is flagged rather than silently rescaled.

The concentration spellings live in `target_units.json`, which step 2 also reads, so the evaluated target cannot drift between the split and the normalisation.

In [ ]:
print(subprocess.run(
    ["python3", "-I", str(RES / "normalise_units.py"), str(V2), str(V2 / "normalised")],
    capture_output=True, text=True, check=True).stdout)

5,807 of 13,054 labels (44.5%) carry a named target. Four points worth recording.

The brief described melting data as 751 CD curve points against 151 real Tm values. Both figures reproduce, and the 151 splits further: 13 are physically impossible, including a 401.1 in the `°C` column and an 8.8 in the `K` column, leaving 138 usable. The `fraction of folded protein` rows are excluded by design, since a point on a denaturation curve is not a melting temperature.

Mass loss is kept separate from relative activity although both are percentages, because 490 `weight loss` labels are a different quantity from activity and the v1 card treated the percent spellings as one thing.

For 369 labels the unit `% relative activity` is more reliable than the variable, which names an assay such as pNP release instead. The unit decides, and the substrate stays on the row, since relative activity on pNP is not the same measurement as on PET.

The PET substrate variants are a judgement call that the data settles. `gfPET`, `pcPET`, `hcPET`, `3PET`, `PET-02mer` and the rest are PET in some form but differ in morphology and oligomer length. Including them adds 129 human-curated labels and every one falls in train, so they cannot improve the evaluation and are left as optional training augmentation.

In [ ]:
norm = read_csv(V2 / "normalised", "labels_normalised.csv")
unassigned = collections.Counter(row["unit_key"] for row in norm if not row["target"])

print(f"unassigned: {sum(unassigned.values()):,} labels over "
      f"{len(unassigned)} unit spellings")
print("what is left, and why it stays out:")
for unit, n in unassigned.most_common(8):
    print(f"  {unit!r:34} {n:5}")

The 7,247 unassigned labels are genuinely heterogeneous rather than a gap to be closed: CD curve points, percentages on variables that name neither activity nor mass loss, concentrations measured on substrates other than PET, thermodynamic quantities in kcal/mol, specific activities in U/mg, pre-normalised rates in µg product per mg PET, and amounts in mmol and µmol that need an assay volume. Closing more of this would mean inventing conversions the references do not support.

### Step 3b: the corrected card and the v2 push

`DATASET_CARD_v2.md` documents the v2 split, the eight targets and the flags, and corrects three claims the v1 card makes on a public repo: the homology guarantee, the count of usable Tm values, and the recoverable PET measurement count.

`push_v2_to_hf.py` publishes the five configs. It refuses to push if any group at any threshold spans folds, since that is the defect v2 exists to fix. Nothing has been pushed; the dry run below builds every config and contacts nothing.

In [ ]:
# Needs pandas, so this cell wants the module environment:
#   module load python/3.12.4 scipy-stack/2024a arrow/21.0.0
#   source ~/petabite/bin/activate && export PYTHONNOUSERSITE=1
print(subprocess.run(
    ["python3", str(RES / "push_v2_to_hf.py"), str(V2),
     "--repo", "petadex/petai-activity-labels", "--dry-run"],
    capture_output=True, text=True, check=True).stdout)

## Results

| | v1 | v2 |
|---|---|---|
| Sequences | 1,219, 1,137 assigned | 1,137, all assigned |
| Labels | 13,092 | 13,054 |
| Stranded labels | 3,188 | 0 |
| Split unit | `cluster_30` | connected component over all three thresholds |
| Groups spanning folds at 90% / 50% / 30% | 0 / 4 / 0 | 0 / 0 / 0 |
| Evaluated target | binary halo activity, 7 negatives in test | PET monomer release, 89 labels on 16 test sequences |
| LLM share of evaluated labels | 62.9% | 0% |
| Named targets | none, `value` is one column | 8, each with a canonical unit |

Folds, where the evaluated set is human-curated PET concentration labels:

| Fold | Components | Sequences | Labels | Evaluated labels | Evaluated sequences | Halo |
|---|---|---|---|---|---|---|
| train | 168 | 956 | 11,700 | 1,564 | 123 | 197 |
| val | 4 | 130 | 629 | 90 | 27 | 0 |
| test | 12 | 51 | 725 | 89 | 16 | 0 |

Targets, from step 3:

| Target | Labels | Sequences | Canonical unit |
|---|---|---|---|
| `pet_release_conc` | 1,997 | 197 | µM |
| `relative_activity` | 1,208 | 82 | percent |
| `kcat` | 713 | 266 | s-1 |
| `kcat_over_km` | 535 | 230 | M-1 s-1 |
| `km` | 529 | 226 | µM |
| `weight_loss` | 490 | 91 | percent |
| `halo_binary` | 197 | 192 | z vs empty vector |
| `melting_temperature` | 138 | 68 | °C |

Both v1 blockers are addressed, though not in the way the plan expected. The negative deficit is resolved by abandoning the target that caused it rather than by rebalancing folds, because no rebalancing exists. The provenance inversion is resolved by defining the evaluated set as human-curated only.

## What this costs, stated plainly

The evaluated test set is 89 labels on 16 sequences. That is small, and it is the honest ceiling for a homology-controlled evaluation of this label set. Three consequences for the plan.

1. A single fixed split at this size gives a wide confidence interval. The 18 components carrying evaluable labels would support k-fold cross-validation over components, pooling 179 labels on 43 sequences, which is the same homology guarantee with roughly double the evaluation data and a bootstrap confidence interval over components as step 6 already wants. Step 5 of the plan specifies one fixed split, so this is a recommendation rather than a change made here.
2. Binary halo activity can be trained but not claimed. Any halo number quoted on the wiki has to be labelled as training-set performance, because 187 of its 197 labels are in one homology component.
3. Train is 84% of sequences. That is a consequence of the component structure rather than a choice, and it does mean effective training diversity is closer to 168 components than 956 sequences, which sharpens the cluster-size asymmetry already noted in step 6 of the plan.

## Next

Steps 1 through 3 are done. Steps 4 through 8 run on `v2`.

Step 4, the dense tracks, is one third ready. Biopython 1.86 is in the `petabite` venv so the ProtParam track runs now. `hmmer/3.4` is available as a module, but the 45 PETadex HMM profiles are not on disk: `petadex-corpus/full/dims/hmms.parquet` holds only their metadata, and its filenames are the `_260507` release, which confirms the HMM release mismatch recorded as blocker 6. SignalP 6 is not available at all; the only module is `signalp/4.1f`, whose output fields differ from the corpus `signalp6` track, so substituting it would break the inference-time parity that track exists to provide. For step 8, `mmseqs2/18-8cc5c` and `diamond/2.1.22` are both available.

`DATASET_CARD_v2.md` and `push_v2_to_hf.py` are written and dry-run clean. Nothing has been pushed to the Hub. The v1 repo is public and its card overstates the homology guarantee, so the card correction is worth making whether or not the v2 data goes up.

Licensing is still marked unsettled on an already-public repo, and that is unchanged by this issue.
